# Chapter 14 &mdash; The Language Families of Turing Machines: RE and Recursive

**Concept 3 of the Chapter 14 decomposition:** *The Language Families of Turing Machines: RE and Recursive*

RE = language of a TM; recursive = language of a TM that halts on every input.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-RE-And-Recursive-Families/Concept-RE-And-Recursive-Families.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.Def_TM         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Two families, defined by what the machine **guarantees**:

* $L$ is **RE** (Turing-recognizable) if some TM $M$ has $L(M)=L$. On a member it
  halts and accepts. On a non-member it may reject **or run forever**.
* $L$ is **recursive** (decidable) if some TM $M$ has $L(M)=L$ **and halts on every
  input**.

The asymmetry is the whole story. An RE machine's silence is uninformative: you cannot
tell "no" from "not yet".

Recursive $\subseteq$ RE, and the inclusion is **strict** &mdash; $A_{TM}$ is the witness
(Concept 10).

## 2. Definitions

### The same language, recognised two ways

In [ ]:
Decide = md2mc('''TM
!! halts on EVERY input
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> D
I : . ; . , R -> D
''')
Recognize = md2mc('''TM
!! same language, but LOOPS on non-members
I : 1 ; 1 , R -> F
I : 0 ; 0 , R -> L
I : . ; . , R -> L
L : 0 ; 0 , R -> L
L : 1 ; 1 , R -> L
L : . ; . , R -> L
''')

# --- thin wrappers over Jove's TM runner --------------------------------
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

### Observing a machine, the only way you can

In [ ]:
def observe(T, tape, fuels=(5, 20, 100, 400)):
    return [(f, 'HALTED' if tm_halts(T, tape, fuel=f) else 'not yet') for f in fuels]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch14&nbsp;2.&nbsp;Procedure vs. Algorithm, and the First Impossibility Result](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Procedure-Vs-Algorithm/Concept-Procedure-Vs-Algorithm.ipynb) &nbsp;&middot;&nbsp; [**Chapter 14** index](https://github.com/ganeshutah/Jove/blob/master/Chapter14/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;4.&nbsp;Language versus Language Family: the Spray Paint Analogy](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Language-Versus-Family/Concept-Language-Versus-Family.ipynb)&nbsp;&rarr;

---

## 3. Tests

Same language.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(1, 5) for p in product('01', repeat=k)]
assert all(tm_accepts(Decide, s, fuel=300) == tm_accepts(Recognize, s, fuel=300)
           for s in strs)
print("both accept exactly the strings starting with 1, on all %d tested" % len(strs))

**Different guarantees.** The decider always answers; the recogniser may not.

In [ ]:
print("on '1' (a member):")
print("   decider   :", observe(Decide, '1'))
print("   recogniser:", observe(Recognize, '1'))
print("on '0' (a non-member):")
print("   decider   :", observe(Decide, '0'))
print("   recogniser:", observe(Recognize, '0'))
assert tm_halts(Decide, '0', fuel=20)
assert not tm_halts(Recognize, '0', fuel=400)

**Silence is uninformative.** You cannot tell 'no' from 'not yet'.

In [ ]:
print("The recogniser on '0' has said nothing after 400 steps.")
print("Is the answer 'no', or is it 'wait'?  From outside, these look identical.")
print()
print("That is the entire practical content of 'RE but not recursive'.")

So the language is **recursive** &mdash; because a decider for it exists.

In [ ]:
print("L = { w : w starts with 1 }")
print("  RE?        yes -- Recognize witnesses it")
print("  recursive? yes -- Decide witnesses it")
print()
print("A language is recursive as soon as SOME halting machine exists,")
print("even if you also have non-halting ones for it.")

The families, and where the strictness comes from.

In [ ]:
print("recursive  SUBSET OF  RE  SUBSET OF  all languages")
print()
print("strict at the first step : A_TM  (Concept 10)")
print("strict at the second     : counting -- countably many TMs,")
print("                           uncountably many languages (Concept 11)")

## 4. Exercises


1. Give an RE language that is obviously recursive. Give one that is not obviously either.
2. Is the complement of a recursive language recursive? Of an RE language?
3. Why is "halts on every input" a property of the **machine**, not the language?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter14/Concept-RE-And-Recursive-Families')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')